# Final Report — VisionServeX v2.32.0

This notebook reads every task's `reports/status.json` and produces
the consolidated final report with quality scan.

In [1]:
import json, sys, os
from pathlib import Path

# Add shared dir to path
NB_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/home/arash/PycharmProjects/VisionServeX/notebook")
sys.path.insert(0, str(NB_ROOT / "shared"))
os.chdir(str(NB_ROOT.parent))

from paths import COCO_400_ANN, COCO_400_IMAGES, SMOKE_IMG, SMOKE_ANN, NB_ROOT, REPO_ROOT
from display import clean, scan_text
from commands import run
from notebook_utils import write_status

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import visionservex
print(f"VisionServeX {visionservex.__version__}")

TASK = NB_ROOT / '99_final_report'


VisionServeX 2.35.0


In [2]:
# Collect all task status files
task_dirs = [d for d in NB_ROOT.iterdir() if d.name[:2].isdigit() and d.is_dir()]
statuses = {}
for td in sorted(task_dirs):
    s_file = td / "reports/status.json"
    if s_file.exists():
        statuses[td.name] = json.loads(s_file.read_text())

print(f"Task status files found: {len(statuses)}")
for task, s in sorted(statuses.items()):
    print(f"  {task:40s}: {s.get('status','?')}")


Task status files found: 12
  01_object_detection                     : benchmark_passed
  02_automatic_segmentation               : benchmark_passed
  03_promptable_segmentation              : benchmark_passed
  04_open_vocab_vlm                       : ?
  05_classification                       : ?
  06_embedding_similarity                 : ?
  07_medical                              : ?
  08_agriculture                          : ?
  09_aerial_obb                           : ?
  10_anomaly_industrial                   : ?
  11_surveillance_video_live              : ?
  12_libreyolo                            : ?


In [3]:
# Coverage ledger from master output
ledger_path = NB_ROOT / "archive_legacy/outputs/visionservex_master_outputs/final/model_coverage_ledger.csv"
if ledger_path.exists():
    ledger = pd.read_csv(ledger_path)
    total = len(ledger)
    n_smoke = (ledger["final_state"]=="smoke_passed").sum()
    n_bench = (ledger["final_state"]=="benchmark_passed").sum()
    n_blocked = ledger["final_state"].isin(["expected_blocker","dependency_required",
                                             "download_failed_retryable","license_blocked",
                                             "manual_checkpoint_required"]).sum()
    print(f"Total registry models: {total}")
    print(f"  smoke_passed        : {n_smoke}")
    print(f"  benchmark_passed    : {n_bench}")
    print(f"  blocked             : {n_blocked}")
    print(f"  unaccounted         : {total - n_smoke - n_bench - n_blocked}")
    ledger.to_csv(TASK/"reports/model_coverage_ledger.csv", index=False)
else:
    print("Coverage ledger not found; re-check visionservex models health-json")


Total registry models: 119
  smoke_passed        : 70
  benchmark_passed    : 0
  blocked             : 49
  unaccounted         : 0


In [4]:
# Quality scan on task outputs
from display import FORBIDDEN
hits = []
for td in sorted(task_dirs):
    for f in (td / "reports").glob("*.json"):
        try:
            text = f.read_text(errors="ignore")
            for n in FORBIDDEN:
                if n in text:
                    hits.append({"file": str(f.relative_to(NB_ROOT)), "needle": n})
        except Exception:
            pass

quality = {
    "strict_forbidden_count": len(hits),
    "forbidden_items": hits[:20],
    "tasks_with_status": len(statuses),
    "status": "PASS" if not hits else "FAIL",
}
(TASK / "reports/quality_scan.json").write_text(json.dumps(quality, indent=2))
print(f"Quality scan: {quality['status']}  ({len(hits)} hits)")


Quality scan: PASS  (0 hits)


In [5]:
# Final winners summary
det_src = NB_ROOT / "01_object_detection/reports/detection_leaderboard.csv"
seg_src = NB_ROOT / "02_automatic_segmentation/reports/segmentation_leaderboard.csv"

winners = {}
if det_src.exists():
    det = pd.read_csv(det_src)
    raw_det = pd.read_csv(NB_ROOT / ".." / "reports/detection_leaderboard_400_v227_source.csv") if (NB_ROOT / ".." / "reports/detection_leaderboard_400_v227_source.csv").exists() else det
    if len(raw_det) and "mAP50_95" in raw_det.columns:
        b = raw_det.sort_values("mAP50_95",ascending=False).iloc[0]
        winners["detection_best_overall"] = f"{b['model_id']} ({b['mAP50_95']:.4f})"
        vsx = raw_det[raw_det["source_engine"]=="visionservex"]
        if len(vsx): winners["detection_best_vsx"] = f"{vsx.iloc[0]['model_id']} ({vsx.iloc[0]['mAP50_95']:.4f})"

if seg_src.exists():
    seg = pd.read_csv(seg_src)
    seg_n = seg.dropna(subset=["mask_mAP50_95"])
    if len(seg_n):
        b = seg_n.sort_values("mask_mAP50_95",ascending=False).iloc[0]
        winners["segmentation_best_overall"] = f"{b['model_id']} ({b['mask_mAP50_95']:.4f})"
        vsx_s = seg_n[seg_n["source_engine"]=="visionservex"]
        if len(vsx_s): winners["segmentation_best_vsx"] = f"{vsx_s.iloc[0]['model_id']} ({vsx_s.iloc[0]['mask_mAP50_95']:.4f})"

(TASK / "reports/final_winners.json").write_text(json.dumps(winners, indent=2))
for k, v in winners.items():
    print(f"  {k:40s}: {v}")
print("\nFinal report complete.")


  detection_best_overall                  : yolo26x.pt (0.4894)
  detection_best_vsx                      : dfine-x-o365-coco (0.4576)
  segmentation_best_overall               : yolo26x-seg.pt (0.2728)
  segmentation_best_vsx                   : rfdetr-seg-medium (0.1011)

Final report complete.
